In [0]:

import json, io
import numpy as np
import pandas as pd

from dataclasses import dataclass
from pyspark.sql import DataFrame
from pyspark.sql.functions import regexp_extract, col
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, FloatType


In [0]:

@dataclass
class IngestConfig:
    # Base paths
    chunks_glob: str                 # e.g., abfss://.../optimized_HH_chunks/*/chunks.jsonl
    emb_npy_glob: str                # e.g., abfss://.../optimized_HH_embeddings/DMRetriever-33M/*/embeddings.npy
    emb_ids_glob: str                # e.g., abfss://.../optimized_HH_embeddings/DMRetriever-33M/*/chunk_ids.json
    
    # Delta tables
    tbl_chunks: str                  # chunk text table
    tbl_emb: str                     # embeddings table
    tbl_rag: str                     # rag table (text + emb)
    
    # Regex for extracting "source_dir" from path
    # NOTE: Must capture the directory name between model dir and filename
    source_dir_regex_npy: str         # regex for embeddings.npy
    source_dir_regex_ids: str         # regex for chunk_ids.json


In [0]:

def read_chunks(chunks_glob: str) -> DataFrame:
    # Read chunks.jsonl and standardize columns
    return (
        spark.read.json(chunks_glob)
        .select("chunk_id", "source_file", "chunk_index_in_file", "text")
    )

def read_binary_with_source_dir(path_glob: str, source_dir_regex: str, content_col: str) -> DataFrame:
    # Read binary files and extract source_dir from file path
    return (
        spark.read.format("binaryFile")
        .load(path_glob)
        .withColumn("source_dir", regexp_extract(col("path"), source_dir_regex, 1))
        .select("source_dir", col("content").alias(content_col))
    )

def pair_embeddings(df_npy: DataFrame, df_ids: DataFrame) -> DataFrame:
    # Join npy and ids by source_dir
    return df_npy.join(df_ids, on="source_dir", how="inner").select("source_dir", "npy_bytes", "ids_bytes")


In [0]:

EMB_SCHEMA = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("embedding", ArrayType(FloatType()), False),
])

def decode_emb_pairs(df_pair: DataFrame, schema: StructType = EMB_SCHEMA) -> DataFrame:
    # Decode embeddings.npy + chunk_ids.json inside executors
    def _decode(pdf_iter):
        for pdf in pdf_iter:
            out = []
            for _, r in pdf.iterrows():
                ids = json.loads(bytes(r["ids_bytes"]).decode("utf-8"))
                vecs = np.load(io.BytesIO(bytes(r["npy_bytes"])))  # [n, dim]
                out.extend((cid, v.astype("float32").tolist()) for cid, v in zip(ids, vecs))
            yield pd.DataFrame(out, columns=["chunk_id", "embedding"])
    return df_pair.mapInPandas(_decode, schema=schema)


In [0]:

def merge_insert_only(df: DataFrame, target_table: str, on: str, insert_cols: list[str], temp_view: str):
    # Register temp view
    df.createOrReplaceTempView(temp_view)
    
    cols_sql = ", ".join(insert_cols)
    vals_sql = ", ".join([f"s.{c}" for c in insert_cols])
    
    spark.sql(f"""
    MERGE INTO {target_table} t
    USING {temp_view} s
    ON t.{on} = s.{on}
    WHEN NOT MATCHED THEN
      INSERT ({cols_sql})
      VALUES ({vals_sql})
    """)


In [0]:

def ingest_one(cfg: IngestConfig) -> dict:
    # 1) chunks
    df_chunks = read_chunks(cfg.chunks_glob)
    merge_insert_only(
        df=df_chunks,
        target_table=cfg.tbl_chunks,
        on="chunk_id",
        insert_cols=["chunk_id", "source_file", "chunk_index_in_file", "text"],
        temp_view="tmp_chunks_in"
    )
    
    # 2) embeddings (read binary + pair + decode)
    df_npy = read_binary_with_source_dir(cfg.emb_npy_glob, cfg.source_dir_regex_npy, "npy_bytes")
    df_ids = read_binary_with_source_dir(cfg.emb_ids_glob, cfg.source_dir_regex_ids, "ids_bytes")
    df_pair = pair_embeddings(df_npy, df_ids)
    df_emb = decode_emb_pairs(df_pair)

    merge_insert_only(
        df=df_emb,
        target_table=cfg.tbl_emb,
        on="chunk_id",
        insert_cols=["chunk_id", "embedding"],
        temp_view="tmp_emb_in"
    )
    
    # 3) rag = inner join (only rows that have both)
    df_rag = df_chunks.join(df_emb, on="chunk_id", how="inner")
    merge_insert_only(
        df=df_rag,
        target_table=cfg.tbl_rag,
        on="chunk_id",
        insert_cols=["chunk_id", "source_file", "chunk_index_in_file", "text", "embedding"],
        temp_view="tmp_rag_in"
    )
    
    # 4) quick stats
    out = {
        "new_source_chunks_glob": cfg.chunks_glob,
        "new_chunks_read": df_chunks.count(),
        "new_emb_decoded": df_emb.count(),
        "new_rag_inner": df_rag.count(),
        "total_chunks_table": spark.table(cfg.tbl_chunks).count(),
        "total_emb_table": spark.table(cfg.tbl_emb).count(),
        "total_rag_table": spark.table(cfg.tbl_rag).count(),
    }
    return out


# Use case

In [0]:
new = "HH"
base = "abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/"

cfg_hh = IngestConfig(
    chunks_glob = base + f"optimized_{new}_chunks/*/chunks.jsonl",
    emb_npy_glob = base + f"optimized_{new}_embeddings/DMRetriever-33M/*/embeddings.npy",
    emb_ids_glob = base + f"optimized_{new}_embeddings/DMRetriever-33M/*/chunk_ids.json",
    tbl_chunks = "tdis_dev_data_catalog.tdir.optimized_chunks_text",
    tbl_emb    = "tdis_dev_data_catalog.tdir.optimized_embeddings_dmretriever33m",
    tbl_rag    = "tdis_dev_data_catalog.tdir.optimized_rag_chunks",
    source_dir_regex_npy = r"/DMRetriever-33M/([^/]+)/embeddings\.npy$",
    source_dir_regex_ids = r"/DMRetriever-33M/([^/]+)/chunk_ids\.json$",
)

stats = ingest_one(cfg_hh)
stats

In [0]:
new = "Qs"
base = "abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/"

cfg_hh = IngestConfig(
    chunks_glob = base + f"optimized_{new}_chunks/*/chunks.jsonl",
    emb_npy_glob = base + f"optimized_{new}_embeddings/DMRetriever-33M/*/embeddings.npy",
    emb_ids_glob = base + f"optimized_{new}_embeddings/DMRetriever-33M/*/chunk_ids.json",
    tbl_chunks = "tdis_dev_data_catalog.tdir.optimized_chunks_text",
    tbl_emb    = "tdis_dev_data_catalog.tdir.optimized_embeddings_dmretriever33m",
    tbl_rag    = "tdis_dev_data_catalog.tdir.optimized_rag_chunks",
    source_dir_regex_npy = r"/DMRetriever-33M/([^/]+)/embeddings\.npy$",
    source_dir_regex_ids = r"/DMRetriever-33M/([^/]+)/chunk_ids\.json$",
)

stats = ingest_one(cfg_hh)
stats

In [0]:

from pyspark.sql.functions import col

new = "Qs"
base = "abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/"
qs_chunks_path = base + f"optimized_{new}_chunks/*/chunks.jsonl"

TBL_CHUNKS = "tdis_dev_data_catalog.tdir.optimized_chunks_text"
TBL_EMB    = "tdis_dev_data_catalog.tdir.optimized_embeddings_dmretriever33m"
TBL_RAG    = "tdis_dev_data_catalog.tdir.optimized_rag_chunks"

df_qs_chunks = spark.read.json(qs_chunks_path).select("chunk_id").distinct()

t_chunks = spark.table(TBL_CHUNKS).select("chunk_id").distinct()
t_emb    = spark.table(TBL_EMB).select("chunk_id").distinct()
t_rag    = spark.table(TBL_RAG).select("chunk_id").distinct()

miss_chunks = df_qs_chunks.join(t_chunks, "chunk_id", "left_anti").count()
miss_emb    = df_qs_chunks.join(t_emb,    "chunk_id", "left_anti").count()
miss_rag    = df_qs_chunks.join(t_rag,    "chunk_id", "left_anti").count()

print("Qs missing in TBL_CHUNKS:", miss_chunks)
print("Qs missing in TBL_EMB   :", miss_emb)
print("Qs missing in TBL_RAG   :", miss_rag)


In [0]:
# COMMAND ----------
from pyspark.sql import functions as F

new = "Qs"
base = "abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/"
qs_chunks_path = base + f"optimized_{new}_chunks/*/chunks.jsonl"

TBL_RAG = "tdis_dev_data_catalog.tdir.optimized_rag_chunks"

qs_ids = spark.read.json(qs_chunks_path).select("chunk_id").distinct()

df = (qs_ids.join(spark.table(TBL_RAG), "chunk_id", "inner")
      .select("chunk_id", "source_file", "chunk_index_in_file", "text")
      .orderBy(F.rand())
      .limit(20))

display(df)


In [0]:
# COMMAND ----------
from pyspark.sql import functions as F

qs = spark.read.json(qs_chunks_path)
files = (qs.select("source_file").where(F.col("source_file").isNotNull()).distinct().orderBy("source_file"))
display(files)


In [0]:
# COMMAND ----------
target_file = "/net/cven-mosta-nas.engr.tamu.edu/volume1/NASdata1/TDIS_PDF/txts used for Qs/SFP-2024 (1).txt"

df = (spark.table(TBL_RAG)
      .where(F.col("source_file") == target_file)
      .select("chunk_id", "chunk_index_in_file", "text")
      .orderBy("chunk_index_in_file"))

display(df.limit(10))
